# Eval analysis: where the judge gets fooled

This notebook loads an eval report and surfaces the cases where the
LLM-as-judge disagreed with ground truth. It uses only the standard library so
it runs anywhere, with no extra install. If the report does not exist yet, the
first cell generates it in mock mode.

The story to look for: the judge returns `PASS` on answers the golden data says
should not pass, and the `groundedness` column shows why for some of them, while
for others (the conflicting-documents case) even groundedness looks fine. No
single signal catches everything.

In [ ]:
import json, os, sys
from pathlib import Path

# Make the repo importable when running from notebooks/.
REPO = Path.cwd()
if (REPO / 'evals').exists() is False and (REPO.parent / 'evals').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

report_path = REPO / 'evals' / 'reports' / 'latest.json'
if not report_path.exists():
    from evals.run_evals import compute_report, _build_results, write_report
    write_report(compute_report(_build_results()))

report = json.loads(report_path.read_text())
print('Loaded report with', len(report['results']), 'tickets')

In [ ]:
# Headline metrics.
m = report['metrics']
def pct(x):
    return f"{round(x*100)}%"
print('Route accuracy:    ', pct(m['route_accuracy']['value']))
print('Answer relevance:  ', pct(m['answer_relevance']['value']))
print('Groundedness:      ', pct(m['groundedness']['value']))
print('Policy compliance: ', pct(m['policy_compliance']['value']))
print('Avg cost/request:   $%.4f' % m['cost_latency']['avg_cost_usd'])
print('Failed cases:      ', report['failed_cases'])

In [ ]:
# The fooled-judge cases, side by side with groundedness.
rows = report['fooled_judge_cases']
header = f"{'ticket':<16}{'judge':<7}{'judge_score':<13}{'groundedness':<14}{'why'}"
print(header)
print('-' * len(header))
for r in rows:
    print(f"{r['id']:<16}{r['judge_verdict']:<7}{str(r['judge_score']):<13}{str(r['groundedness']):<14}{r['why']}")

print()
print('Read this as: the judge said PASS on all of these, but ground truth says')
print('they should not pass. Groundedness catches some (missing-001, badjudge-001)')
print('but not conflicting-001, whose answer is well grounded in the wrong source.')

In [ ]:
# Full per-ticket view for context.
cols = ['id', 'route', 'expected_route', 'decision', 'expected_decision', 'judge_verdict', 'cost']
print(''.join(f'{c:<18}' for c in cols))
for r in report['results']:
    print(''.join(f'{str(r.get(c)):<18}' for c in cols))

## Takeaway

The LLM-as-judge is the eval people reach for first because it is easy to build
and reads as objective. On this dataset it passes four answers that are wrong,
ungrounded, or empty. A validated judge, checked against golden data and paired
with deterministic metrics like groundedness and policy compliance, is what turns
an eval you *trust* into an eval you can *verify*.